In [14]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from tqdm.auto import tqdm
from datetime import datetime

from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.metrics import f1_score, balanced_accuracy_score


from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [15]:
def calculate_business_metrics(y_true, y_pred) -> dict[str, float]:
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

    luxury_recall = float(report.get("2", {}).get("recall", 0.0))

    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)

    severe = ((y_true == 0) & (y_pred == 2)) | ((y_true == 2) & (y_pred == 0))
    severe_rate = float(severe.mean())

    return {
        "luxury_recall": luxury_recall,
        "severe_misclassification_rate": severe_rate,
    }

In [16]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, current_year=None):
        self.current_year = current_year

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()

        year = pd.to_numeric(X.get("yearOfRegistration"), errors="coerce")
        km = pd.to_numeric(X.get("kilometer"), errors="coerce")
        power = pd.to_numeric(X.get("power"), errors="coerce")

        cy = self.current_year or datetime.now().year
        age = cy - year

        X["vehicleAge"] = age
        X["kmPerYear"] = km / (age + 1)
        X["km_power_ratio"] = km / (power + 1)
        X["log_km"] = np.log1p(km)

        return X.drop(columns=["yearOfRegistration"], errors="ignore")


class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=self.cols, errors="ignore")

class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols: list[str]):
        self.cols = cols

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        self.freq_maps_ = {}

        for col in self.cols:
            if col in X_df.columns:
                self.freq_maps_[col] = X_df[col].value_counts(normalize=True)
            else:
                self.freq_maps_[col] = pd.Series(dtype=float)

        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()

        for col in self.cols:
            freq = self.freq_maps_.get(col)

            if col in X_df.columns:
                X_df[f"{col}_freq"] = X_df[col].map(freq).fillna(0.0)
            else:
                X_df[f"{col}_freq"] = 0.0

        return X_df

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            make_column_selector(dtype_include=np.number),
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first")),
                ]
            ),
            make_column_selector(dtype_include=object),
        ),
    ],
    remainder="drop",
)

In [17]:
df = pd.read_csv("../data/processed/clean_data.csv")

df["price_tier"] = (
    df["price_tier"]
    .astype(str)
    .str.lower()
    .map({"budget": 0, "mid-range": 1, "luxury": 2})
)

df = df.dropna(subset=["price_tier"])

X = df.drop(columns=["price_tier", "price"], errors="ignore")
y = df["price_tier"].astype(int)

In [18]:
fe = FeatureEngineer(current_year=datetime.now().year)
freq = FrequencyEncoder(cols=["brand", "model"])
dropper = DropColumns(["brand", "model"])

X = fe.fit_transform(X)
X = freq.fit_transform(X)
X = dropper.fit_transform(X)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=42
)

X_train = pd.get_dummies(X_train, drop_first=True)
X_val = pd.get_dummies(X_val, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [19]:
# SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# OR undersampling
rus = RandomUnderSampler(random_state=42)
X_train_us, y_train_us = rus.fit_resample(X_train, y_train)

In [20]:
DATASETS = {
    "none": (X_train, y_train),
    "smote": (X_train_smote, y_train_smote),
    "undersample": (X_train_us, y_train_us),
}

In [ ]:
model_configs = {
    "baseline": {
        "estimator": DummyClassifier(strategy="most_frequent"),
        "params": {},
        "n_iter": 1,
    },
    "random_forest": {
        "estimator": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators": [100, 200, 300],
            "max_depth": [None, 10, 20, 30],
            "min_samples_split": [2, 5, 10],
        },
        "n_iter": 10,
    },
    "extra_trees": {
        "estimator": ExtraTreesClassifier(random_state=42),
        "params": {
            "n_estimators": [100, 200, 300],
            "max_depth": [None, 10, 20],
            "min_samples_split": [2, 5, 10],
        },
        "n_iter": 10,
    },
    "log_reg": {
        "estimator": LogisticRegression(solver="saga", max_iter=5000),
        "params": {
            "C": np.logspace(-2, 1, 5),
        },
        "n_iter": 5,
    },
    "xgboost": {
        "estimator": XGBClassifier(
            objective="multi:softprob",
            num_class=3,
            tree_method="hist",
            n_jobs=1,
            random_state=42
        ),
        "params": {
            "n_estimators": [300, 500, 800],
            "max_depth": [4, 6, 8, 10],
            "learning_rate": [0.01, 0.05, 0.1],
            "subsample": [0.7, 0.8, 1.0],
            "colsample_bytree": [0.7, 0.8, 1.0],
        },
        "n_iter": 20
    },
    "decision_tree": {
        "estimator": DecisionTreeClassifier(random_state=42),
        "params": {
            "max_depth": [None, 5, 10, 20, 30],
            "min_samples_split": [2, 5, 10, 20],
            "min_samples_leaf": [1, 2, 4, 8],
            "criterion": ["gini", "entropy"],
            "max_features": [None, "sqrt", "log2"],
        },
        "n_iter": 10,
    },
    "linear_svm": {
        "estimator": LinearSVC(random_state=42, max_iter=5000),
        "params": {
            "C": np.logspace(-3, 2, 6),
            "loss": ["hinge", "squared_hinge"],
        },
        "n_iter": 10,
    },
}

In [ ]:
def run_model(model_name: str, dataset_type: str, use_fs: bool = False):

    cfg = model_configs[model_name]

    X_tr, y_tr = DATASETS[dataset_type]

    run_name = f"{model_name}_{dataset_type}"

    print(f"\nRunning: {run_name}")

    best_model = None
    best_score = -1
    best_params = None

    param_list = list(
        ParameterSampler(cfg["params"], n_iter=cfg["n_iter"], random_state=42)
    )

    with mlflow.start_run(run_name=run_name):

        # =========================
        # SEARCH
        # =========================
        for params in param_list:

            model = cfg["estimator"].set_params(**params)

            model.fit(X_tr, y_tr)

            val_pred = model.predict(X_val)

            score = f1_score(y_val, val_pred, average="macro")

            if score > best_score:
                best_score = score
                best_model = model
                best_params = params

        # =========================
        # TEST EVALUATION
        # =========================
        test_pred = best_model.predict(X_test)

        test_f1 = f1_score(y_test, test_pred, average="macro")
        test_bal = balanced_accuracy_score(y_test, test_pred)

        # =========================
        # BUSINESS METRICS (ADDED BACK)
        # =========================
        biz_metrics = calculate_business_metrics(y_test, test_pred)

        # =========================
        # MLflow logging
        # =========================
        mlflow.log_param("model", model_name)
        mlflow.log_param("dataset_type", dataset_type)
        mlflow.log_params(best_params)

        mlflow.log_metric("val_f1", best_score)
        mlflow.log_metric("test_f1", test_f1)
        mlflow.log_metric("test_balanced_acc", test_bal)

        mlflow.log_metric("luxury_recall", biz_metrics["luxury_recall"])
        mlflow.log_metric(
            "severe_misclassification_rate",
            biz_metrics["severe_misclassification_rate"],
        )

        mlflow.sklearn.log_model(best_model, "model")

    print(f"Done: {run_name} | val_f1={best_score:.4f} | test_f1={test_f1:.4f}")

    return {
        "model": model_name,
        "dataset": dataset_type,
        "val_f1": best_score,
        "test_f1": test_f1,
        "test_bal_acc": test_bal,
        "luxury_recall": biz_metrics["luxury_recall"],
        "severe_misclassification_rate": biz_metrics["severe_misclassification_rate"],
    }

In [23]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("used_car_price_tier")

<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1777741693493, experiment_id='2', last_update_time=1777741693493, lifecycle_stage='active', name='used_car_price_tier', tags={}, trace_location=None, workspace='default'>

In [30]:
random_forest_none = run_model("random_forest", "none")
random_forest_smote = run_model("random_forest", "smote")
random_forest_us = run_model("random_forest", "undersample")


Running: random_forest_none


2026/05/02 23:14:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 23:14:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run random_forest_none at: http://127.0.0.1:5000/#/experiments/2/runs/706d6f996fdd4b938cb06dd9af21a6db
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Done: random_forest_none | val_f1=0.8502 | test_f1=0.8257

Running: random_forest_smote


2026/05/02 23:18:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 23:18:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run random_forest_smote at: http://127.0.0.1:5000/#/experiments/2/runs/ef5dfe5a5eb445c39ed6649ec1c4625e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Done: random_forest_smote | val_f1=0.8476 | test_f1=0.8238

Running: random_forest_undersample


2026/05/02 23:20:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 23:20:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run random_forest_undersample at: http://127.0.0.1:5000/#/experiments/2/runs/ca1ed2ae368f44f7a73b77feec84548e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Done: random_forest_undersample | val_f1=0.8414 | test_f1=0.8217


In [31]:
xgboost_none = run_model("xgboost", "none")
xgboost_smote = run_model("xgboost", "smote")
xgboost_us = run_model("xgboost", "undersample")


Running: xgboost_none


2026/05/02 23:23:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 23:23:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run xgboost_none at: http://127.0.0.1:5000/#/experiments/2/runs/3a65fe9887154d08b45052af80360b66
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Done: xgboost_none | val_f1=0.8591 | test_f1=0.8590

Running: xgboost_smote


2026/05/02 23:26:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 23:26:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run xgboost_smote at: http://127.0.0.1:5000/#/experiments/2/runs/6eb1728422514647afd53421b034f27d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Done: xgboost_smote | val_f1=0.8560 | test_f1=0.8560

Running: xgboost_undersample


2026/05/02 23:27:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 23:27:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run xgboost_undersample at: http://127.0.0.1:5000/#/experiments/2/runs/15cea5bc8c944016973ae1bbbe258b27
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Done: xgboost_undersample | val_f1=0.8516 | test_f1=0.8529


In [26]:
log_reg_none = run_model("log_reg", "none")
log_reg_smote = run_model("log_reg", "smote")
log_reg_us = run_model("log_reg", "undersample")


Running: log_reg_none


2026/05/02 22:03:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 22:03:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run log_reg_none at: http://127.0.0.1:5000/#/experiments/2/runs/227db1f66d7843a1b5fda428ac34714b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Done: log_reg_none | val_f1=0.6694 | test_f1=0.6606

Running: log_reg_smote


2026/05/02 22:25:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 22:25:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run log_reg_smote at: http://127.0.0.1:5000/#/experiments/2/runs/b785bd852972489d910126bb9b5db4ad
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Done: log_reg_smote | val_f1=0.6654 | test_f1=0.6617

Running: log_reg_undersample


2026/05/02 22:34:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 22:34:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run log_reg_undersample at: http://127.0.0.1:5000/#/experiments/2/runs/a4dcad63015d4c1faaaa65c664ffd933
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Done: log_reg_undersample | val_f1=0.6644 | test_f1=0.6600


In [28]:
decision_tree_none = run_model("decision_tree", "none")
decision_tree_smote = run_model("decision_tree", "smote")
decision_tree_us = run_model("decision_tree", "undersample")


Running: decision_tree_none


2026/05/02 22:40:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 22:40:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run decision_tree_none at: http://127.0.0.1:5000/#/experiments/2/runs/f1d72c7cf13c4975bdaed21df8718a0c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Done: decision_tree_none | val_f1=0.8339 | test_f1=0.8296

Running: decision_tree_smote


2026/05/02 22:40:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 22:40:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run decision_tree_smote at: http://127.0.0.1:5000/#/experiments/2/runs/e3e8243039d840e3b97599efb6390958
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Done: decision_tree_smote | val_f1=0.8316 | test_f1=0.8254

Running: decision_tree_undersample


2026/05/02 22:40:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 22:40:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run decision_tree_undersample at: http://127.0.0.1:5000/#/experiments/2/runs/2160aa16cfeb4f10b89c489034e9b58a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Done: decision_tree_undersample | val_f1=0.8215 | test_f1=0.8240


In [29]:
baseline_none = run_model("baseline", "none")
baseline_smote = run_model("baseline", "smote")
baseline_us = run_model("baseline", "undersample")


Running: baseline_none


2026/05/02 22:43:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 22:43:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run baseline_none at: http://127.0.0.1:5000/#/experiments/2/runs/56b1d39904884ca49839bb31f089c7af
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Done: baseline_none | val_f1=0.2074 | test_f1=0.2074

Running: baseline_smote


2026/05/02 22:43:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 22:43:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run baseline_smote at: http://127.0.0.1:5000/#/experiments/2/runs/7e8b4528c17c42098b2a2dada7841709
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Done: baseline_smote | val_f1=0.2074 | test_f1=0.2074

Running: baseline_undersample


2026/05/02 22:43:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 22:43:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run baseline_undersample at: http://127.0.0.1:5000/#/experiments/2/runs/151100daba7644199238bee4297d7a50
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Done: baseline_undersample | val_f1=0.2074 | test_f1=0.2074


In [32]:
all_results = []
all_results.append(xgboost_none)
all_results.append(xgboost_smote)
all_results.append(xgboost_us)

all_results.append(log_reg_none)
all_results.append(log_reg_smote)
all_results.append(log_reg_us)

all_results.append(decision_tree_none)
all_results.append(decision_tree_smote)
all_results.append(decision_tree_us)

all_results.append(random_forest_none)
all_results.append(random_forest_smote)
all_results.append(random_forest_us)

all_results.append(baseline_none)

In [33]:
all_results

[{'model': 'xgboost',
  'dataset': 'none',
  'val_f1': 0.8590912805430443,
  'test_f1': 0.8590372905869573,
  'test_bal_acc': 0.8566052899720881,
  'luxury_recall': 0.8611006051437217,
  'severe_misclassification_rate': 0.0021999652637063626},
 {'model': 'xgboost',
  'dataset': 'smote',
  'val_f1': 0.8559842328938934,
  'test_f1': 0.8559772383568793,
  'test_bal_acc': 0.8580324067229896,
  'luxury_recall': 0.8856845688350984,
  'severe_misclassification_rate': 0.0025473282000810513},
 {'model': 'xgboost',
  'dataset': 'undersample',
  'val_f1': 0.8516456427343044,
  'test_f1': 0.8528869696685698,
  'test_bal_acc': 0.8586011346376335,
  'luxury_recall': 0.9053517397881997,
  'severe_misclassification_rate': 0.0028367973137266253},
 {'model': 'log_reg',
  'dataset': 'none',
  'val_f1': 0.6693829995980121,
  'test_f1': 0.660631166631072,
  'test_bal_acc': 0.661593184023632,
  'luxury_recall': 0.6931732223903178,
  'severe_misclassification_rate': 0.02115054323703661},
 {'model': 'log_reg'

In [36]:
results_df = pd.DataFrame(all_results)

results_df = results_df.sort_values("test_f1", ascending=False)
results_df.to_csv("../reports/model_comparison.csv", index=False)
print(results_df)

            model      dataset    val_f1   test_f1  test_bal_acc  \
0         xgboost         none  0.859091  0.859037      0.856605   
1         xgboost        smote  0.855984  0.855977      0.858032   
2         xgboost  undersample  0.851646  0.852887      0.858601   
6   decision_tree         none  0.833869  0.829641      0.826029   
9   random_forest         none  0.850171  0.825669      0.819681   
7   decision_tree        smote  0.831617  0.825440      0.825583   
8   decision_tree  undersample  0.821532  0.823992      0.827864   
10  random_forest        smote  0.847568  0.823785      0.828110   
11  random_forest  undersample  0.841403  0.821727      0.827556   
4         log_reg        smote  0.665410  0.661730      0.678718   
3         log_reg         none  0.669383  0.660631      0.661593   
5         log_reg  undersample  0.664373  0.660021      0.677673   
12       baseline         none  0.207449  0.207449      0.333333   

    luxury_recall  severe_misclassification_rat

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from mlflow.tracking import MlflowClient

client = MlflowClient()

experiment = client.get_experiment_by_name("used_car_price_tier")


run_id = "3a65fe9887154d08b45052af80360b66"
model = mlflow.sklearn.load_model(f"runs:/{run_id}/model")
y_pred = model.predict(X_test)

In [45]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix

with open(f"best_model_classification_report.txt", "w") as f:
    f.write(classification_report(y_test, y_pred, zero_division=0))

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("best_model")

plt.savefig(f"best_model_confusion_matrix.png", bbox_inches="tight")
plt.close()